# Feature Engineering – CredIntel

## Objective

The objective of this notebook is to create meaningful
financial indicators from the loan application dataset.

These features help analyze borrower financial strength,
loan affordability, asset coverage, and credit risk.

This notebook focuses on data transformation and
business-oriented financial analysis using Python.

In [39]:
import pandas as pd
import numpy as np

df = pd.read_csv("loan_approval_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Clean text values
text_columns = df.select_dtypes(include='object').columns

for col in text_columns:
    df[col] = df[col].str.strip()

df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


## Feature 1: Total Assets

Total Assets represent the overall financial strength of an applicant by combining residential, commercial, luxury, and bank assets.


In [24]:
df['total_assets'] = (
    df['residential_assets_value']
    + df['commercial_assets_value']
    + df['luxury_assets_value']
    + df['bank_asset_value']
)

df[['loan_id','total_assets']].head()

,loan_id,total_assets
0,1,50700000
1,2,17000000
2,3,57700000
3,4,52700000
4,5,55000000


## Feature 2: Loan-to-Income Ratio

This metric measures how large the requested loan is relative to the applicant's annual income.
Higher values may indicate increased repayment risk.

In [25]:
df['loan_to_income_ratio'] = (
    df['loan_amount']
    / df['income_annum']
)

df[['loan_id','loan_to_income_ratio']].head()

,loan_id,loan_to_income_ratio
0,1,3.114583
1,2,2.975610
2,3,3.263736
3,4,3.743902
4,5,2.469388


## Feature 3: Asset-to-Loan Ratio

This metric evaluates the applicant's financial coverage relative to the requested loan amount.
Higher values indicate stronger collateral support.

In [26]:
df['asset_to_loan_ratio'] = (
    df['total_assets']
    / df['loan_amount']
)

df[['loan_id','asset_to_loan_ratio']].head()

,loan_id,asset_to_loan_ratio
0,1,1.695652
1,2,1.393443
2,3,1.942761
3,4,1.716612
4,5,2.272727


In [27]:
df[['total_assets',
    'loan_to_income_ratio',
    'asset_to_loan_ratio']].describe()

,total_assets,loan_to_income_ratio,asset_to_loan_ratio
count,4.269000e+03,4269.000000,4269.000000
mean,3.254877e+07,2.984807,2.231797
std,1.950656e+07,0.595496,0.643145
min,4.000000e+05,1.500000,0.750000
25%,1.630000e+07,2.464286,1.767347
50%,3.150000e+07,3.000000,2.142857
75%,4.720000e+07,3.500000,2.616216
max,9.070000e+07,4.000000,5.666667


## Feature 4: Credit Risk Score

A custom risk score is created using the applicant's credit score.

This score helps classify customers into High Risk, Medium Risk, and Low Risk segments for lending decisions.

In [28]:
df['risk_category'] = pd.cut(
    df['cibil_score'],
    bins=[300, 550, 700, 900],
    labels=['High Risk', 'Medium Risk', 'Low Risk']
)

df['risk_category'].value_counts()

risk_category
High Risk      1778
Low Risk       1425
Medium Risk    1055
Name: count, dtype: int64

## Feature 5: Financial Strength Category

Customers are segmented based on total asset ownership.

In [29]:
df['financial_strength'] = pd.qcut(
    df['total_assets'],
    q=3,
    labels=['Low Assets', 'Medium Assets', 'High Assets']
)

df['financial_strength'].value_counts()

financial_strength
Low Assets       1428
Medium Assets    1422
High Assets      1419
Name: count, dtype: int64

### Feature 5: Financial Strength Category

Applicants are segmented into three groups based on
their total asset values using quantile-based grouping.

The categories represent relative asset ownership
within this dataset and do not represent official
bank lending eligibility thresholds.

## Feature 6: Borrowing Capacity

Borrowing capacity is determined using the Loan-to-Income Ratio.

In [30]:
df['borrowing_capacity'] = pd.cut(
    df['loan_to_income_ratio'],
    bins=[0,2,3,5],
    labels=['Strong', 'Moderate', 'Aggressive']
)

df['borrowing_capacity'].value_counts()

borrowing_capacity
Moderate      2093
Aggressive    2056
Strong         120
Name: count, dtype: int64

In [31]:
pd.crosstab(
    df['risk_category'],
    df['loan_status']
)

loan_status,Approved,Rejected
risk_category,,
High Risk,186,1592
Medium Risk,1049,6
Low Risk,1418,7


In [37]:
conditions = [
    df['cibil_score'] < 550,
    (df['cibil_score'] >= 550) & (df['cibil_score'] < 700),
    df['cibil_score'] >= 700
]

categories = ['High Risk', 'Medium Risk', 'Low Risk']

df['risk_category'] = np.select(
    conditions,
    categories,
    default='Unknown'
)

df[['cibil_score', 'risk_category']].head()

,cibil_score,risk_category
0,778,Low Risk
1,417,High Risk
2,506,High Risk
3,467,High Risk
4,382,High Risk


In [35]:
pd.crosstab(
    df['financial_strength'],
    df['loan_status']
)

loan_status,Approved,Rejected
financial_strength,,
Low Assets,905,523
Medium Assets,881,541
High Assets,870,549


## Key Business Insights

1. CIBIL-based risk segmentation shows a strong
   association with loan approval outcomes.

2. Total asset ownership helps evaluate the
   relative financial position of applicants.

3. Loan-to-income ratio provides an indicator
   of requested loan size relative to annual income.

4. Asset-to-loan ratio measures total asset coverage
   relative to the requested loan amount.

5. These engineered features support descriptive
   credit risk and lending portfolio analysis.